# Proyecto final · Análisis de actividad antifúngica

**Guía para estudiantes.**

Trabajamos con los ensayos *in vitro* de inhibición del crecimiento micelial y con la
caracterización DLS de las formulaciones, contenidos en
`resultados actividad antifúngica.xlsx`.

| Bloque | Producto |
|---|---|
| 1. Conexión con los datos | El Excel cargado y verificado en pandas |
| 2. Skills | El paquete completo de `scientific-agent-skills` instalado en `.agents/skills/` |
| 3. Limpieza | Tablas *tidy* de actividad y de DLS |
| 4. Descriptiva | Medias, desvíos y auditoría de faltantes por grupo |
| 5. Bioestadística | Inhibición, supuestos, tests, dosis-respuesta 4PL y sensibilidad |
| 6. Visualización | Figuras guardadas en `outputs/` |
| 7. IR | Bloque preparado para las formulaciones ≥ 8 (datos aún no disponibles) |

**Cómo trabajar con este cuaderno.** En la guía, cada bloque trae una consigna, una
pista y un espacio para anotar el prompt; las celdas de código están vacías. Acá están
los prompts de referencia y las soluciones, con las mismas decisiones metodológicas.

**Advertencias que atraviesan todo el análisis.**

- El control de crecimiento tiene **una sola medición** (1.9 cm): todos los `% de
  inhibición` son condicionales a ese valor y su incertidumbre no es estimable.
- Las 3 celdas `NA` son faltantes declarados; los 5 ceros son **inhibición completa**,
  un dato válido que se evalúa en el análisis de sensibilidad.
- Solo 2 dosis en T5–T8: no permite estimar una curva dosis-respuesta creíble.
- El análisis es **exploratorio**: los valores de *p* se reportan como descriptivos.


## 0. Configuración

Esta es la única celda que nombra archivos, hojas y constantes. El resto del
cuaderno usa nombres genéricos (`tidy`, `dls_tidy`, `x`, `y`). Para trabajar con
otro archivo, se edita acá y se vuelve a ejecutar todo.


**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 1. Conectar a la IA con la base de datos de antifúngicos

**Producto:** el Excel leído en pandas, con sus dos hojas identificadas y verificadas.

La "conexión" consiste en poner los datos en un DataFrame que el asistente pueda
leer y discutir. No se copia ni se edita el archivo original: se registra su
SHA-256 para saber que siempre trabajamos sobre la misma versión.


### Consigna 1.1 · Leer el archivo

Cargá el Excel con `pd.ExcelFile`, listá las hojas y lee cada una en un DataFrame.
Guardá la hoja de actividad en `actividad_raw` y la de DLS en `dls_raw` (esta última
sin encabezado, porque tiene dos filas de títulos).

**Se espera:** la lista de hojas y las formas `(filas, columnas)` de los dos DataFrames.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

`pd.ExcelFile(RUTA_EXCEL).sheet_names` da los nombres. `pd.read_excel(excel, sheet_name=HOJA_DLS, header=None)` respeta la cabecera doble de DLS.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 1.2 · Verificar antes de seguir

Comprobá con código —no a ojo— que los encabezados de la hoja de actividad son los
esperados y que la huella del archivo coincide con la que se registró en la
configuración.

**Se espera:** un `assert` que pase y el SHA-256 impreso.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Compará `list(actividad_raw.columns)` contra una lista literal y usá la función `sha256` definida arriba.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 2. Descargar un paquete de skills

**Producto:** todas las skills de `K-Dense-AI/scientific-agent-skills` instaladas en
`.agents/skills/`.

El link original del cuaderno (`nazarenacorvalan-sys/Proyecto_final_antifungicos`)
hoy devuelve 404. Se usa en su lugar el repositorio que ya fijó el curso en su
`skills-lock.json`, con el mismo *commit* para que la descarga sea reproducible.


### Consigna 2.1 · Instalar y contar

Descargá el zip del repositorio fijado en el commit `c21d13b…`, extraé **solo** el
directorio `skills/` dentro de `.agents/skills/`, y contá cuántos archivos
`SKILL.md` quedaron. Volver a ejecutar la celda no debe duplicar nada.

**Se espera:** un mensaje con la cantidad de skills instaladas y un `assert` de que no
sea cero.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

El zip de codeload tiene una carpeta raíz tipo scientific-agent-skills-<commit>. Buscala con un set de prefijos en vez de escribirla a mano. `ZipFile.infolist()` permite filtrar las entradas que empiezan con `<raiz>/skills/`.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 3. Limpieza: de celdas fusionadas a tablas *tidy*

La hoja de actividad tiene 20 bloques de tres filas con la etiqueta de tratamiento
en la primera (celdas fusionadas). La hoja DLS tiene dos filas de encabezado,
coma decimal, el símbolo `±` y valores `ND`. Cada hoja necesita su own lectura.


### Consigna 3.1 · Actividad antifúngica

 Propagá la etiqueta hacia las filas en blanco, convertí los `NA` en faltantes,
separá `tratamiento` y `dosis` con una expresión regular y conservá los ceros.
La unidad de análisis es la réplica biológica: una fila por medición.

**Se espera:** un DataFrame `tidy` con las columnas `grupo`, `familia`,
`dosis_ug_ml`, `crecimiento_cm`, `replica_id` y `es_control`, con el control
codificado como dosis 0. Imprimir cantidad de filas, faltantes y ceros.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Usá ffill() sobre la columna de tratamientos y str.strip() para quitar el espacio final de etiquetas como 'T3 40 µg/ml '. La regex puede ser r'^(T\d+)\s+(\d+)\s*µg/ml$'. Los ceros NO son faltantes.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 3.2 · DLS

Parseá la hoja `DLS`: encontrá la primera fila de datos, separá `"792,5 ± 4,9"`
en media y desvío, pasá la coma decimal a punto y convertí `ND` en faltante.

**Se espera:** `dls_tidy` con una fila por formulación (T5–T8), tamaño y desvío en
`t0` y `t30`, y los dos índices de polidispersidad.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

La primera fila de datos se detecta con str.match(r'^T\d+$') sobre la columna 0. Para el tamaño: str(valor).strip().replace(',', '.').split('±'). Usá pd.to_numeric(..., errors='coerce') para los PDI.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 4. Estadística descriptiva

**Producto:** una tabla resumen por grupo y una auditoría de faltantes.

El control se muestra aparte: tiene una sola medición, así que no tiene desvío.


### Consigna 4.1 · Descriptivos y faltantes

Agregá `crecimiento_cm` por grupo con n, media, mediana, desvío, mínimo y máximo,
y agregá el crecimiento relativo al control. Después mostrá cuántas observaciones
y cuántos faltantes hay en cada combinación familia × dosis.

**Se espera:** 20 filas en la tabla de descriptivos. Las proporciones de la auditoría
deben sumar el total de filas de `tidy`.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Excluí el control con ~tidy['es_control']. La media del control es CRECIMIENTO_CONTROL_CM. Para la auditoría usá groupby(...).agg(n='count', faltantes=lambda s: s.isna().sum()).

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 5. Bioestadística

**Producto:** porcentajes de inhibición, decisión de tests por familia, comparaciones
pareadas, curvas dosis-respuesta y análisis de sensibilidad.

Todo se trata como exploratorio. La decisión del test se toma **antes** de mirar
los resultados, usando normalidad sobre los residuos del modelo y homocedasticidad
con Levene centrado en la mediana.


### Consigna 5.1 · Inhibición y supuestos

Calculá `% inhibición = 100·(1 − g/1.9)` y evaluá los supuestos dentro de cada
familia T1–T8. Usá Shapiro-Wilk sobre los residuos (restar la media de cada dosis)
y Levene con centro = mediana entre las dosis.

**Se espera:** una tabla `supuestos` con n, p de Shapiro, p de Levene y las dos
decisiones binarias con `α = 0.05`.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Los residuos son crecimiento_cm menos la media de su propia dosis; quitá los NaN. Para Levene pasá una lista de muestras, una por dosis, con center='median'. Con n < 3, Shapiro no es aplicable: marcá la decisión como None.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 5.2 · Tests por familia

Aplicá la regla anterior: ANOVA + Tukey si hay normalidad y homocedasticidad;
Welch ANOVA + comparaciones de Welch con Holm si solo falla la homocedasticidad;
Kruskal-Wallis + Mann-Whitney con Holm en los demás casos.

**Se espera:** `resultados` con el test ómnibus por familia y `comparaciones` con
cada par de dosis, su *p* cruda, su *p* ajustada y el flag de significancia.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Implementá Welch ANOVA a mano para no depender de módulos extra: pesos w_i = n_i/s_i^2, media ponderada, F con df1=k−1 y df2=(k²−1)/(3·Σ(1−w_i/Σw)²/(n_i−1)). Sumá 1e-12 a las varianzas para evitar división por cero en los grupos de ceros.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 5.3 · Curvas dosis-respuesta (4PL)

Ajustá la logística de cuatro parámetros para T1–T4, con la asíntota superior fija
en el control (1.9 cm). Si el grupo tiene ceros, fijá la inferior en 0; si no,
estimala entre 0 y 1.9. Obtené el IC 95% de la EC50 por bootstrap de réplicas
dentro de cada dosis, con la semilla fija.

**Se espera:** una tabla con EC50, pendiente, R² e IC; `EC50 no estimable` cuando
el ajuste no converge o el IC no es finito. T5–T8 se excluyen por tener solo 2 dosis.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

La función es bottom + (top−bottom)/(1+(EC50/x)^hill). Usá curve_fit con bounds y maxfev alto. En cada bootstrap, remuestreá dentro de cada dosis con rng.integers y volvé a ajustar; si falla más de la mitad de las veces, declará la EC50 no estimable.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 5.4 · Sensibilidad de los ceros

Repetí el resumen de los grupos con ceros bajo dos escenarios: ceros como dato
válido (análisis principal) y ceros reemplazados por ε = 0.01 cm.

**Se espera:** una tabla corta que muestre que la conclusión cualitativa no cambia.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Filtrá T3 80 µg/ml y T4 80 µg/ml. En el segundo escenario, usá .loc para cambiar solo los ceros por 0.01 antes de calcular medias e inhibición.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 6. Visualización

**Producto:** figuras guardadas en `outputs/`, con ejes rotulados y unidades.

Se usan puntos individuales sobre las barras para que se vea la réplica biológica,
barras de error de un desvío estándar y letras de significancia para las dosis de
T1–T4.


### Consigna 6.1 · Figura de inhibición

Graficá el `% de inhibición` medio por grupo con barras de error DE, los puntos
individuales encima y letras compactas derivadas de `comparaciones`. Guardá en
PNG a 300 dpi y en PDF.

**Se espera:** un archivo `outputs/fig1_inhibicion.png` y su versión `.pdf`, con la
barra del control en 0%.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Para las letras, probá letras de la 'a' a la 'h' en orden: una letra se puede asignar a un grupo si ninguno de los grupos que ya la tiene difiere significativamente de él según comparaciones.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 6.2 · Curvas dosis-respuesta

Graficá las curvas 4PL ajustadas en escala logarítmica de dosis, con los puntos
observados y la EC50 anotada. Guardá el resultado en `outputs/fig2_dosis_respuesta.png`.

**Se espera:** una figura con cuatro paneles (T1–T4) y la EC50 impresa en cada uno.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Usá plt.subplots(2, 2, figsize=(9, 7), sharex=True, sharey=True). La curva se evalúa en np.logspace(log10(min), log10(max), 100). Solo anotá EC50 si ajustes_4pl.estimable es True.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 6.3 · DLS en el tiempo

Graficá el tamaño y el índice de polidispersidad en `t0` y `t30` para T5–T8,
mostrando las barras de error de la desviación del tamaño.

**Se espera:** dos paneles, con `ND` visible como ausencia de dato en T7 y T8 a t30.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Pasá dls_tidy a formato largo con pd.melt o armá dos ejes con errorbar. Los valores ND ya son NaN, así que simplemente no se dibujan.

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 7. Espectroscopía IR (bloque preparado)

Faltan los datos de IR de las formulaciones desde la 8 en adelante. Esta celda
busca archivos que coincidan con `PATRON_IR`; si no están, avisa y se salta sin
romper el cuaderno. Cuando lleguen, no hace falta cambiar el resto del análisis.


### Consigna 7.1 · Cargar si están disponibles

Buscá archivos con `Path('.').glob(PATRON_IR)`. Si hay, concatenalos en `datos_ir`;
si no, imprimí un aviso y dejá `datos_ir = None`.

**Se espera:** un mensaje claro sobre la ausencia de datos, sin excepciones.


<details>
<summary><b>¿Te trabaste?</b> (clic para ver una pista)</summary>

Path.glob devuelve un generador; convertilo en lista con sorted(). Para concatenar, pd.concat([pd.read_csv(a) for a en archivos], ignore_index=True).

</details>

**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


## 8. Verificación e informe

**Producto:** checks automáticos de integridad y un resumen escrito en
`outputs/informe_resumen.md`.


### Consigna 8.1 · Chequeos finales

Verificá que no haya negativos, que los faltantes declarados sigan siendo 3, que
los ceros sigan siendo 5, que DLS tenga 4 formulaciones y que las figuras existan.

**Se espera:** `Chequeos: OK` o un `AssertionError` que señale qué cambió.


**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


### Consigna 8.2 · Informe resumen

Escribí un resumen breve con el SHA-256, la estructura de datos, las decisiones de
tests y las advertencias de interpretación.

**Se espera:** `outputs/informe_resumen.md` creado y su contenido impreso.


**Tu prompt:**

*...*

In [ ]:
# Pegá o escribí aquí el código que generó el asistente


#### Tu interpretación

*...*